# Exercice XP — IA agentique avec RAG (Retrieval-Augmented Generation)**Objectif :** construire un mini-agent open source, sans clé API.Composants :1. Base de connaissances en mémoire (FAISS + `FakeEmbeddings`)2. Outil externe gratuit (Wikipedia)3. Planificateur à base de règles4. Fonction de réponse avec citations `[kb:doc1]` / `[wiki:Titre]`5. Vérification rapide sur 3 questions

## 0. Installation

In [ ]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

In [ ]:
import warnings, rewarnings.filterwarnings("ignore")from langchain_core.documents import Documentfrom langchain_community.embeddings import FakeEmbeddingsfrom langchain_community.vectorstores import FAISSfrom langchain_community.chat_models.fake import FakeListChatModelfrom langchain_community.utilities.wikipedia import WikipediaAPIWrapperprint("Imports OK")

## Exercice 1 — Construire le récupérateur de base de connaissances5 à 8 `Document` avec un champ `source`, indexés dans FAISS via `FakeEmbeddings`, retriever top-3.

In [ ]:
KB_DOCS = [    Document(        page_content=(            "Python est un langage de programmation interprete, de haut niveau et generaliste, "            "cree par Guido van Rossum et publie en 1991. Il privilegie la lisibilite du code "            "grace a une indentation significative."        ),        metadata={"source": "doc1", "title": "Python"},    ),    Document(        page_content=(            "RAG (Retrieval-Augmented Generation) combine un moteur de recherche et un modele de "            "langage. On recupere d'abord des passages pertinents, puis le LLM genere une reponse "            "ancree dans ces passages, ce qui reduit les hallucinations."        ),        metadata={"source": "doc2", "title": "RAG"},    ),    Document(        page_content=(            "FAISS est une bibliotheque de Meta AI pour la recherche de similarite vectorielle. "            "Elle indexe des embeddings et retrouve les k plus proches voisins tres rapidement, "            "meme sur des millions de vecteurs."        ),        metadata={"source": "doc3", "title": "FAISS"},    ),    Document(        page_content=(            "LangChain est un framework open source pour construire des applications basees sur "            "des LLM. Il fournit des abstractions : chaines, outils, agents, retrievers et "            "magasins vectoriels."        ),        metadata={"source": "doc4", "title": "LangChain"},    ),    Document(        page_content=(            "Un embedding est une representation vectorielle dense d'un texte. Deux textes de sens "            "proche ont des vecteurs proches selon la similarite cosinus, ce qui permet la "            "recherche semantique."        ),        metadata={"source": "doc5", "title": "Embeddings"},    ),    Document(        page_content=(            "Un agent IA percoit son environnement, planifie une suite d'actions et utilise des "            "outils externes (recherche, calcul, API) pour atteindre un objectif, au lieu de "            "repondre en une seule passe."        ),        metadata={"source": "doc6", "title": "Agent IA"},    ),    Document(        page_content=(            "Wikipedia est une encyclopedie libre et collaborative lancee en 2001. Son API publique "            "permet de recuperer des resumes d'articles sans clef d'authentification."        ),        metadata={"source": "doc7", "title": "Wikipedia"},    ),    Document(        page_content=(            "Une hallucination est une affirmation produite par un LLM qui est fluide mais fausse "            "ou non fondee. Citer ses sources et ancrer la generation dans des documents recuperes "            "est la principale parade."        ),        metadata={"source": "doc8", "title": "Hallucination"},    ),]embeddings = FakeEmbeddings(size=256)vectorstore = FAISS.from_documents(KB_DOCS, embeddings)retriever = vectorstore.as_retriever(search_kwargs={"k": 3})def kb_search(query: str):    """Retourne les 3 meilleurs documents de la base de connaissances."""    docs = retriever.invoke(query)    return [{"source": d.metadata["source"],             "title": d.metadata["title"],             "text": d.page_content} for d in docs]for hit in kb_search("Qu'est-ce que le RAG ?"):    print(f"[kb:{hit['source']}] {hit['title']} -> {hit['text'][:70]}...")

> **Note :** `FakeEmbeddings` genere des vecteurs aleatoires — la pertinence n'est donc pas semantique.> C'est voulu pour l'exercice (aucune clef requise). En production : `OpenAIEmbeddings` ou `HuggingFaceEmbeddings`.

## Exercice 2 — Outil externe gratuit (Wikipedia)

In [ ]:
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=600, lang="fr")def wiki_search(query: str, k: int = 2):    """Retourne une courte liste d'extraits Wikipedia avec leur titre."""    try:        pages = wiki_api.load(query)[:k]    except Exception as e:        print(f"[wiki] erreur: {e}")        return []    out = []    for p in pages:        title = p.metadata.get("title", "Inconnu")        slug = re.sub(r"\s+", "_", title.strip())        out.append({"source": f"wiki:{slug}", "title": title,                    "text": p.page_content[:600].strip()})    return outfor s in wiki_search("Tour Eiffel"):    print(f"[{s['source']}] {s['title']} -> {s['text'][:70]}...")

## Exercice 3 — Planificateur a base de reglesSi la question mentionne un sujet connu de la KB -> `kb`. Sinon -> `wikipedia`.

In [ ]:
KB_TOPICS = {    "python": "doc1", "rag": "doc2", "retrieval": "doc2", "generation augmentee": "doc2",    "faiss": "doc3", "vectoriel": "doc3", "similarite": "doc3",    "langchain": "doc4",    "embedding": "doc5", "vecteur": "doc5",    "agent": "doc6", "agentique": "doc6",    "wikipedia": "doc7",    "hallucination": "doc8",}def plan(question: str) -> dict:    """Planificateur a base de regles -> dictionnaire de plan simple."""    q = question.lower()    matched = sorted({t for t in KB_TOPICS if t in q})    if matched:        return {"question": question, "route": "kb", "tools": ["kb_search"],                "matched_topics": matched,                "reason": f"Sujet(s) couvert(s) par la KB : {', '.join(matched)}"}    return {"question": question, "route": "wikipedia", "tools": ["wiki_search"],            "matched_topics": [],            "reason": "Aucun sujet de la KB detecte -> recours a Wikipedia."}for q in ["Qu'est-ce que FAISS ?", "Qui a peint la Joconde ?", "Parle-moi de ca"]:    print(plan(q), "\n")

## Exercice 4 — Fonction de reponse- Recupere selon le plan, combine contexte KB + extraits wiki- LLM stub `FakeListChatModel` par defaut ; option `sshleifer/tiny-gpt2` en local- Cite toujours `[kb:doc1]` / `[wiki:Titre]`- Signale les preuves minces + propose une question de suivi

In [ ]:
USE_HF = False  # passer a True pour tester le pipeline HF localfake_llm = FakeListChatModel(responses=[    "Voici une synthese fondee sur les elements recuperes ci-dessus.",    "D'apres les sources disponibles, voici la reponse la plus probable.",    "Les extraits recuperes permettent de repondre partiellement.",] * 20)hf_llm = Noneif USE_HF:    from transformers import pipeline    from langchain_community.llms import HuggingFacePipeline    pipe = pipeline("text-generation", model="sshleifer/tiny-gpt2",                    max_new_tokens=40, pad_token_id=50256)    hf_llm = HuggingFacePipeline(pipeline=pipe)def _llm_generate(prompt: str) -> str:    if USE_HF and hf_llm is not None:        return hf_llm.invoke(prompt).strip()    return fake_llm.invoke(prompt).content.strip()MIN_EVIDENCE = 2  # seuil au-dela duquel les preuves ne sont plus jugees "minces"def answer(question: str) -> dict:    p = plan(question)    evidence = []    if p["route"] == "kb":        evidence += kb_search(question)    else:        evidence += wiki_search(question)        if not evidence:  # repli sur la KB si Wikipedia ne renvoie rien            evidence += kb_search(question)            p["tools"].append("kb_search (repli)")    citations = [f"[kb:{e['source']}]" if not e["source"].startswith("wiki:")                 else f"[{e['source']}]" for e in evidence]    if not evidence:        return {"plan": p, "sources": [],                "answer": ("Je n'ai trouve aucune preuve, ni dans la base de connaissances "                           "ni sur Wikipedia. Question de suivi : pouvez-vous preciser le "                           "sujet exact ou reformuler avec un terme plus specifique ?")}    context = "\n\n".join(f"{c} {e['title']} :: {e['text']}"                           for c, e in zip(citations, evidence))    prompt = (f"Question : {question}\n\nContexte :\n{context}\n\n"              "Reponds uniquement a partir du contexte et cite les sources.")    synthesis = _llm_generate(prompt)    lines = [synthesis, "", "Elements retenus :"]    for c, e in zip(citations, evidence):        lines.append(f"  - {c} {e['title']} : {e['text'][:140].rstrip()}...")    thin = len(evidence) < MIN_EVIDENCE or p["route"] == "wikipedia" and not p["matched_topics"] and len(evidence) < 2    if thin:        lines += ["", "Preuves minces : la confiance est faible.",                  f"Question de suivi : pouvez-vous preciser le contexte de « {question} » "                  "(domaine, periode, ou terme exact) ?"]    lines += ["", "Sources : " + ", ".join(citations)]    return {"plan": p, "sources": citations, "answer": "\n".join(lines)}print(answer("Qu'est-ce qu'un embedding ?")["answer"])

## Exercice 5 — Verification rapide (3 questions)

In [ ]:
QUESTIONS = [    "Qu'est-ce que FAISS et a quoi sert-il ?",   # couverte par la KB    "Qui a construit la Tour Eiffel ?",           # externe -> Wikipedia    "Comment ca marche ?",                        # ambigue]for i, q in enumerate(QUESTIONS, 1):    r = answer(q)    print("=" * 78)    print(f"Q{i} : {q}")    print("-" * 78)    print("PLAN    :", r["plan"])    print("SOURCES :", r["sources"] or "aucune")    print("-" * 78)    print("REPONSE :")    print(r["answer"])    print()

## Conclusion| Exercice | Realisation ||---|---|| 1 | 8 `Document` + FAISS/`FakeEmbeddings`, retriever top-3 || 2 | `WikipediaAPIWrapper` -> extraits titres + texte, sans clef || 3 | Planificateur a base de regles renvoyant un dict `{route, tools, matched_topics, reason}` || 4 | Reponse citee `[kb:docN]` / `[wiki:Titre]`, gestion des preuves minces, LLM stub ou HF || 5 | 3 questions : KB / externe / ambigue, avec plan + sources + reponse |**Pistes d'amelioration :** vrais embeddings (`HuggingFaceEmbeddings`), planificateur LLM au lieu de regles, reranking, seuil de score de similarite.